In [1]:
# Cell 1: imports y configuración
import os, json, math, random, time, csv
from pathlib import Path

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm

# nuestros utilitarios previos
import diff_utils as DU
import train_utils as TU  # face_crop_fn, default_preprocess, sample_uniform_frames, read_dfdc_metadata

# helpers recién creados (descargados)
from transformer_utils import ViTWithQualityAndCWT
from ensemble_utils import weighted_blend, StackingHead, make_stacking_features

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ==== Rutas/params ====
ROOT = r"E:\ERIS\Ernesto\dfdc_train_all"          # carpeta con dfdc_train_part_0..49
CKPT_DIFF = r".\checkpoints_dfdc_eps\eps_unet_final.pt"  # checkpoint del prior (ε-UNet)
PARTS_TRAIN = tuple(range(0, 8))
PARTS_VAL   = (8, 9)

T_STEPS = 300
IMAGE_SIZE = 128
CHANNELS   = 3
FRAMES_PER_VIDEO = 8
USE_FACE_CROP = True  # mismo recorte que en training del difusor

# ViT + CWT config
PATCH = 8
D_MODEL = 384
DEPTH   = 6
HEADS   = 6
CWT_SCALES = [5,9,13,17]        # tamaños de kernel gabor
CWT_THETAS = [0, 45, 90, 135]   # orientaciones
CWT_OUT_CH = len(CWT_SCALES) * len(CWT_THETAS)  # canales del stack gabor

BATCH = 4
EPOCHS = 5
LR = 2e-4
WEIGHT_DECAY = 1e-4
AMP = True
GRAD_CLIP = 1.0

random.seed(0); np.random.seed(0); torch.manual_seed(0)


Device: cuda


c:\Users\labdsp\miniconda3\envs\Deepfake\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Cell 2: cargar el prior ε-UNet y armar función para ELBO por frame
diff_model = DU.TinyUNet(in_ch=CHANNELS, base=32, t_dim=128).to(device)
state = torch.load(CKPT_DIFF, map_location=device)
diff_model.load_state_dict(state["model"])
diff_model.eval()

sched = DU.build_schedule(T=T_STEPS, device=device)

@torch.no_grad()
def elbo_per_frame(frame_bgr, image_size=IMAGE_SIZE, channels=CHANNELS):
    # recorte igual que training del prior
    if USE_FACE_CROP:
        try:
            frame_bgr = TU.face_crop_fn(frame_bgr)
        except Exception:
            pass
    x = TU.default_preprocess(frame_bgr, out_size=image_size, to_gray=(channels==1), norm="tanh")
    x = x.unsqueeze(0).to(device)  # [1,C,H,W]
    e = DU.estimate_elbo_mc(diff_model, x, sched).mean().item()
    return float(e), x.squeeze(0).cpu()  # devolvemos también el tensor usado


C:\Users\labdsp\AppData\Local\Temp\ipykernel_15280\2650552304.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(CKPT_DIFF, map_location=device)


In [3]:
# Cell 3: construir un banco de filtros Gabor como proxy de CWT 2D sobre la cara
def gabor_stack_from_gray(gray_u8, scales=CWT_SCALES, thetas=CWT_THETAS):
    # gray_u8: np.uint8 [H,W]
    H, W = gray_u8.shape[:2]
    stack = []
    for k in scales:
        sigma = 0.56 * k
        lam   = k * 2.0
        for theta_deg in thetas:
            theta = np.deg2rad(theta_deg)
            kern = cv2.getGaborKernel((k, k), sigma, theta, lam, gamma=0.5, psi=0, ktype=cv2.CV_32F)
            resp = cv2.filter2D(gray_u8, cv2.CV_32F, kern)
            # normalizamos a [0,1]
            r = cv2.normalize(resp, None, 0, 1, cv2.NORM_MINMAX)
            stack.append(r)
    if len(stack) == 0:
        return np.zeros((1, H, W), np.float32)
    arr = np.stack(stack, axis=0).astype(np.float32)  # [Cg, H, W]
    return arr

def build_cwt_stack_from_bgr(frame_bgr, out_size=IMAGE_SIZE):
    if USE_FACE_CROP:
        try:
            frame_bgr = TU.face_crop_fn(frame_bgr)
        except Exception:
            pass
    # a gris 0-255
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    gray = TU.center_square_crop(gray, ratio=0.9)
    gray = cv2.resize(gray, (out_size, out_size), interpolation=cv2.INTER_AREA)
    stack = gabor_stack_from_gray(gray)  # [Cg,H,W] in [0,1]
    return stack


In [4]:
# Cell 4: Dataset para entrenar la Transformer con token de calidad + rama CWT
class DFDCFusedDataset(Dataset):
    def __init__(self, root, parts, frames_per_video=8, max_videos=None, split_filter=None):
        """
        split_filter: "train", "val" o None (usa metadato 'split' del JSON si existe)
        """
        self.root = Path(root)
        self.frames_per_video = frames_per_video
        self.items = []  # [(video_path, label_int)]
        for p in parts:
            pd = self.root / f"dfdc_train_part_{p}"
            if not pd.is_dir(): 
                continue
            meta = TU.read_dfdc_metadata(pd)
            for fname, rec in meta.items():
                lab = str(rec.get("label","")).upper()
                if lab not in ("REAL", "FAKE"): 
                    continue
                if split_filter is not None:
                    if str(rec.get("split","")).lower() != split_filter:
                        continue
                vpath = pd / fname
                if vpath.is_file():
                    y = 1 if lab=="FAKE" else 0
                    self.items.append((str(vpath), y))
        if max_videos:
            self.items = self.items[:max_videos]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        vpath, y = self.items[idx]
        cap = cv2.VideoCapture(vpath)
        if not cap.isOpened():
            # fallback: tensor vacío -> zeros
            frames = torch.zeros(self.frames_per_video, 1, IMAGE_SIZE, IMAGE_SIZE)
            e_bpd  = torch.zeros(self.frames_per_video)
            cwt    = torch.zeros(self.frames_per_video, CWT_OUT_CH, IMAGE_SIZE, IMAGE_SIZE)
            return frames, e_bpd, cwt, torch.tensor(y, dtype=torch.long), Path(vpath).name

        idxs = TU.sample_uniform_frames(cap, want=self.frames_per_video)
        frames, e_list, cwt_list = [], [], []
        for fi in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
            ok, frame = cap.read()
            if not ok:
                # fallback a negro
                frames.append(torch.zeros(1, IMAGE_SIZE, IMAGE_SIZE))
                e_list.append(0.0)
                cwt_list.append(np.zeros((CWT_OUT_CH, IMAGE_SIZE, IMAGE_SIZE), np.float32))
                continue
            # ELBO + tensor preprocess (misma normalización que prior)
            e, x = elbo_per_frame(frame)             # e: float, x: [C,H,W] in [-1,1]
            frames.append(x.cpu())
            e_list.append(e)
            # CWT/Gabor stack
            cwt_stack = build_cwt_stack_from_bgr(frame)  # [Cg,H,W] in [0,1]
            cwt_list.append(cwt_stack)

        cap.release()
        frames = torch.stack(frames, dim=0)   # [T,C,H,W]
        e_bpd  = torch.tensor(e_list, dtype=torch.float32)  # [T]
        cwt    = torch.tensor(np.stack(cwt_list, axis=0), dtype=torch.float32)  # [T,Cg,H,W]

        return frames, e_bpd, cwt, torch.tensor(y, dtype=torch.long), Path(vpath).name


In [5]:
# Cell 5: DataLoaders
train_ds = DFDCFusedDataset(ROOT, PARTS_TRAIN, frames_per_video=FRAMES_PER_VIDEO, split_filter="train")
val_ds   = DFDCFusedDataset(ROOT, PARTS_VAL,   frames_per_video=FRAMES_PER_VIDEO, split_filter="val")

train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=4, pin_memory=True)
print("Train videos:", len(train_ds), "Val videos:", len(val_ds))


Train videos: 16357 Val videos: 0


In [6]:
# Cell 6: Modelo y optimizador
model = ViTWithQualityAndCWT(
    img_size=IMAGE_SIZE,
    in_chans=CHANNELS,
    patch_size=PATCH,
    d_model=D_MODEL,
    depth=DEPTH,
    heads=HEADS,
    cwt_in_chans=CWT_OUT_CH,
    cwt_dim=256,
    num_classes=2,
).to(device)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type=="cuda") and AMP)
ce = nn.CrossEntropyLoss(reduction='none')  # para ponderar por calidad si quieres

def quality_weights_from_elbo(e_bpd_batch, eps=1e-6):
    """
    Genera pesos por video a partir de T valores de ELBO (menor ELBO = mejor calidad).
    Aquí usamos w = 1 / (eps + mean(e_bpd)), normalizado en el batch.
    """
    w = 1.0 / (eps + e_bpd_batch.mean(dim=1))  # [B]
    w = w / (w.mean() + 1e-6)
    return w


C:\Users\labdsp\AppData\Local\Temp\ipykernel_15280\2580625644.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type=="cuda") and AMP)


In [ ]:
# Cell 7: train/val loops
from math import isfinite

def run_epoch(dl, train=True):
    model.train(mode=train)
    total_loss, total_correct, total_count = 0.0, 0, 0
    all_logits, all_labels = [], []

    for frames, e_bpd, cwt, y, _ in tqdm(dl, desc="train" if train else "valid", ncols=0):
        frames = frames.to(device)       # [B,T,C,H,W] in [-1,1]
        e_bpd  = e_bpd.to(device)        # [B,T], libre escala (ELBO)
        cwt    = cwt.to(device)          # [B,T,Cg,H,W] in [0,1]
        y      = y.to(device)            # [B]

        with torch.cuda.amp.autocast(enabled=(device.type=="cuda") and AMP):
            logits, aux = model(frames, e_bpd, cwt)  # [B,2]
            # pesos por calidad:
            w = quality_weights_from_elbo(e_bpd)     # [B]
            loss_vec = ce(logits, y) * w
            loss = loss_vec.mean()

        if train:
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if GRAD_CLIP is not None:
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(opt)
            scaler.update()

        total_loss += loss.item() * frames.size(0)
        pred = logits.argmax(dim=1)
        total_correct += (pred == y).sum().item()
        total_count += frames.size(0)

        all_logits.append(logits.detach().cpu())
        all_labels.append(y.detach().cpu())

    avg_loss = total_loss / max(1, total_count)
    acc = total_correct / max(1, total_count)
    logits = torch.cat(all_logits, dim=0) if all_logits else torch.empty(0,2)
    labels = torch.cat(all_labels, dim=0) if all_labels else torch.empty(0, dtype=torch.long)
    return avg_loss, acc, logits, labels

best_val = 0.0
for ep in range(1, EPOCHS+1):
    tr_loss, tr_acc, _, _ = run_epoch(train_dl, train=True)
    va_loss, va_acc, va_logits, va_labels = run_epoch(val_dl, train=False)
    print(f"[Epoch {ep}/{EPOCHS}] train_loss={tr_loss:.4f} acc={tr_acc:.3f} | val_loss={va_loss:.4f} acc={va_acc:.3f}")
    if va_acc > best_val:
        best_val = va_acc
        torch.save({"model": model.state_dict(), "epoch": ep}, "vit_qcwt_best.pt")
        print("  -> saved vit_qcwt_best.pt")


train:   0% 0/4089 [00:00<?, ?it/s]

In [ ]:
# Cell 8: Ensemble sobre el split de validación
best = torch.load("vit_qcwt_best.pt", map_location=device)
model.load_state_dict(best["model"])
model.eval()

def eval_with_ensemble(dl, alpha=5.0):
    all_probs_vit, all_probs_ens, all_y = [], [], []
    all_e_stats = []

    for frames, e_bpd, cwt, y, _ in tqdm(dl, desc="ensemble-val", ncols=0):
        frames = frames.to(device)
        e_bpd  = e_bpd.to(device)
        cwt    = cwt.to(device)
        y      = y.to(device)

        with torch.no_grad():
            logits, aux = model(frames, e_bpd, cwt)   # [B,2]
            prob_vit = F.softmax(logits, dim=-1)[:,1] # FAKE

            # diff prob heurística a partir de ELBO (menor ELBO ~ mayor calidad real)
            e_mean = aux["e_stats"][:,0]  # [B]
            # construimos un "score fake" más alto cuando ELBO medio es alto:
            # prob_diff = sigmoid(alpha * (e_mean - ref)), con ref= mediana del batch (podrías usar global)
            ref = e_mean.median()
            prob_diff = torch.sigmoid(alpha*(e_mean - ref))

            # cwt prob: si no tienes cabeza separada, usa prob_vit como proxy o agrega una proyección:
            prob_cwt = torch.sigmoid((aux["z_cwt"] @ torch.zeros(aux["z_cwt"].size(1), device=device)).sum(dim=1))
            # (arriba es dummy=0; mejor entrena una cabeza CWT-only si quieres un verdadero prob_cwt)

            prob_ens = weighted_blend(prob_vit, prob_cwt, prob_diff, weights=(0.7, 0.1, 0.2))

        all_probs_vit.append(prob_vit.cpu())
        all_probs_ens.append(prob_ens.cpu())
        all_y.append(y.cpu())
        all_e_stats.append(aux["e_stats"].cpu())

    pv = torch.cat(all_probs_vit); pe = torch.cat(all_probs_ens); yy = torch.cat(all_y)
    est = torch.cat(all_e_stats)

    def acc_from_probs(p, y, thr=0.5):
        pred = (p >= thr).long()
        return (pred == y).float().mean().item()

    acc_vit = acc_from_probs(pv, yy)
    acc_ens = acc_from_probs(pe, yy)
    print(f"VAL acc_vit={acc_vit:.3f} | acc_ens={acc_ens:.3f}")
    return pv, pe, yy, est

pv, pe, yy, est = eval_with_ensemble(val_dl)


In [ ]:
# Cell 9: Stacking opcional (logistic)
stacker = StackingHead().to(device)
opt_s = torch.optim.AdamW(stacker.parameters(), lr=1e-3, weight_decay=1e-4)

def train_stacker(dl, epochs=3):
    stacker.train()
    for ep in range(1, epochs+1):
        losses = []
        for frames, e_bpd, cwt, y, _ in tqdm(dl, desc=f"stacker ep{ep}", ncols=0):
            frames = frames.to(device); e_bpd = e_bpd.to(device); cwt = cwt.to(device); y = y.float().to(device)
            with torch.no_grad():
                logits, aux = model(frames, e_bpd, cwt)
                prob_vit = F.softmax(logits, dim=-1)[:,1]
                # dummy prob_cwt (ver comentario anterior)
                prob_cwt = torch.sigmoid((aux["z_cwt"] @ torch.zeros(aux["z_cwt"].size(1), device=device)).sum(dim=1))
                feats = make_stacking_features(logits, prob_cwt, aux["e_stats"])  # [B,5]
            pred = stacker(feats)
            loss = F.binary_crossentropy_with_logits if False else F.binary_cross_entropy(pred, y)
            opt_s.zero_grad(set_to_none=True); loss.backward(); opt_s.step()
            losses.append(loss.item())
        print(f"stacker ep{ep} loss={np.mean(losses):.4f}")
    stacker.eval()

# train_stacker(train_dl)


In [ ]:
# Cell 10: muestra algunos recortes y su ELBO usado
import matplotlib.pyplot as plt

def show_samples(ds, k=6):
    idxs = np.random.choice(len(ds), size=min(k, len(ds)), replace=False)
    fig, axes = plt.subplots(len(idxs), FRAMES_PER_VIDEO, figsize=(FRAMES_PER_VIDEO*2.2, len(idxs)*2.2))
    if len(idxs) == 1: 
        axes = np.expand_dims(axes, 0)
    for r, ii in enumerate(idxs):
        frames, e_bpd, cwt, y, name = ds[ii]
        for c in range(min(FRAMES_PER_VIDEO, frames.size(0))):
            ax = axes[r, c]
            img = frames[c,0].numpy()
            ax.imshow((img+1)/2, cmap="gray", vmin=0, vmax=1)
            ax.set_title(f"e={e_bpd[c]:.3f}")
            ax.axis("off")
        axes[r,0].set_ylabel(f"{name}\nlabel={'FAKE' if y.item()==1 else 'REAL'}")
    plt.tight_layout()

show_samples(val_ds, k=3)
